# Extension Experiments for the HAR Comparison Study (Random Forest, Gradient Boosting, Repeated and Leave-One-Subject-Out CV, Subject-Level Bootstrap)

**Run this notebook AFTER the main pipeline notebook.** It does *not* repeat the hyperparameter grid searches of the
four original models (SVM-Linear, SVM-RBF, KNN, Decision Tree): their best hyperparameters from the main run
(`config_and_environment.json`) are fixed below. Only the two new ensemble models are tuned (small grid,
subject-grouped 5-fold CV on the training subjects only; the test subjects are never used for tuning).

| Section | Content | Addresses the reviewer question |
|---|---|---|
| A | Two ensemble baselines (Random Forest, Gradient Boosting) added to the comparison on the held-out test subjects | "Why are no ensemble methods included?" |
| B | Subject-level (cluster) bootstrap confidence intervals on the test set | "Bootstrap CIs treat correlated windows as independent" |
| C | Repeated subject-grouped K-fold CV on all 30 subjects (several random seeds) | "Results depend on a single split" |
| D | Leave-one-subject-out (LOSO) evaluation, per-subject accuracy | "Only 9 test subjects / no LOSO" |
| E | Confusion matrices and per-class results saved as raw tables | (figure for the manuscript) |
| F | Download all outputs as one ZIP | |

All outputs are in English and numbered `Ext..` so they do not clash with the main notebook outputs.

## 0. Setup

In [ ]:
!pip install -q kagglehub

In [ ]:
import os, time, json, shutil, warnings, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, scipy
from scipy import stats
try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut, GridSearchCV
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef,
                             confusion_matrix, precision_recall_fscore_support)
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# ------------------------- USER SETTINGS -------------------------
SEED = 42
FAST_MODE = False          # True: quick trial run. Leave False for manuscript-quality results.
N_FOLDS = 3 if FAST_MODE else 10          # folds of the repeated grouped CV
N_REPEATS = 1 if FAST_MODE else 3         # number of random repetitions (different seeds)
N_BOOT = 200 if FAST_MODE else 2000       # subject-level bootstrap resamples
RF_TREES = 50 if FAST_MODE else 300
HGB_ITER = 30 if FAST_MODE else 200
RUN_REPEATED_CV = True
RUN_LOSO = True            # leave-one-subject-out (about 15-25 min on Colab); set False to skip
# ----------------------------------------------------------------
np.random.seed(SEED)
OUT_DIR = Path("har_extension_outputs"); (OUT_DIR / "tables").mkdir(parents=True, exist_ok=True); (OUT_DIR / "figures").mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight"})
T0 = time.time()

def save_table(df, name, caption=""):
    df.to_csv(OUT_DIR / "tables" / f"{name}.csv")
    try:
        df.to_latex(OUT_DIR / "tables" / f"{name}.tex", float_format="%.4f", caption=caption or None, label=f"tab:{name}")
    except Exception:
        pass

def savefig(name):
    plt.savefig(OUT_DIR / "figures" / f"{name}.png"); plt.savefig(OUT_DIR / "figures" / f"{name}.pdf"); plt.show()

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def mcnemar_p(a_correct, b_correct):
    n01 = int(np.sum(a_correct & ~b_correct)); n10 = int(np.sum(~a_correct & b_correct))
    if n01 + n10 == 0: return n01, n10, 1.0
    chi2 = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)   # McNemar with continuity correction
    return n01, n10, float(stats.chi2.sf(chi2, 1))

## 1. Data Loading

In [ ]:
import kagglehub

path = kagglehub.dataset_download("uciml/human-activity-recognition-with-smartphones")
csv_files = {p.name.lower(): p for p in Path(path).rglob("*.csv")}
if not {"train.csv", "test.csv"} <= set(csv_files):
    raise FileNotFoundError(f"train.csv / test.csv not found. Available: {list(csv_files)}")
train_df = pd.read_csv(csv_files["train.csv"]); test_df = pd.read_csv(csv_files["test.csv"])
print("Train:", train_df.shape, "| Test:", test_df.shape)

In [ ]:
TARGET, GROUP = "Activity", "subject"
feature_cols = [c for c in train_df.columns if c not in (TARGET, GROUP)]
assert not (set(train_df[GROUP]) & set(test_df[GROUP])), "Train/test subjects overlap!"

CLASS_EN = {"WALKING": "Walking", "WALKING_UPSTAIRS": "Walking Upstairs", "WALKING_DOWNSTAIRS": "Walking Downstairs",
            "SITTING": "Sitting", "STANDING": "Standing", "LAYING": "Lying"}
le = LabelEncoder().fit(train_df[TARGET]); K = len(le.classes_)
class_names = [CLASS_EN.get(c, c) for c in le.classes_]

X_train = train_df[feature_cols].to_numpy(float); y_train = le.transform(train_df[TARGET]); g_train = train_df[GROUP].to_numpy()
X_test = test_df[feature_cols].to_numpy(float);   y_test = le.transform(test_df[TARGET]);   g_test = test_df[GROUP].to_numpy()
X_all = np.vstack([X_train, X_test]); y_all = np.concatenate([y_train, y_test]); g_all = np.concatenate([g_train, g_test])
print("Subjects: train", len(np.unique(g_train)), "| test", len(np.unique(g_test)), "| total", len(np.unique(g_all)))

## 2. Models

The four original models use the best hyperparameters found by the main notebook (copied from
`config_and_environment.json`). The two ensemble models are tuned in Section A.

In [ ]:
def pipe(clf):
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

MODELS = {
    "SVM (Linear)":  pipe(SVC(kernel="linear", C=0.1, random_state=SEED)),
    "SVM (RBF)":     pipe(SVC(kernel="rbf", C=100, gamma=0.001, random_state=SEED)),
    "KNN":           pipe(KNeighborsClassifier(n_neighbors=21, weights="distance", metric="manhattan")),
    "Decision Tree": pipe(DecisionTreeClassifier(criterion="entropy", max_depth=12, min_samples_leaf=5, random_state=SEED)),
}
ORIGINAL = list(MODELS)

# Published test accuracies from the main run (Table05), used as a reproducibility check
EXPECTED_TEST_ACC = {"SVM (Linear)": 0.9617, "SVM (RBF)": 0.9586, "KNN": 0.9060, "Decision Tree": 0.8517}

## A. Ensemble Baselines on the Held-Out Test Subjects

Random Forest and Gradient Boosting (histogram-based) are tuned with the same subject-grouped 5-fold protocol
(`StratifiedGroupKFold`, macro-F1) on the **training subjects only**, then evaluated once on the 9 test subjects.

In [ ]:
cv5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
ens = {
    "Random Forest": (pipe(RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=SEED)),
                      {"clf__max_features": ["sqrt", 0.2], "clf__min_samples_leaf": [1, 3]}),
    "Gradient Boosting": (pipe(HistGradientBoostingClassifier(max_iter=HGB_ITER, random_state=SEED)),
                          {"clf__learning_rate": [0.05, 0.1], "clf__l2_regularization": [0.0, 1.0]}),
}
tune_rows = []
for name, (p, grid) in ens.items():
    t = time.time()
    gs = GridSearchCV(p, grid, scoring="f1_macro", cv=cv5, n_jobs=1 if name == "Random Forest" else -1, refit=False)
    gs.fit(X_train, y_train, groups=g_train)
    best = gs.best_params_
    MODELS[name] = clone(p).set_params(**best)
    i = gs.best_index_
    tune_rows.append({"Model": name, "Best Hyperparameters": ", ".join(f"{k.replace('clf__','')}={v}" for k, v in best.items()),
                      "CV Macro-F1 (mean +/- std)": f"{gs.cv_results_['mean_test_score'][i]:.4f} +/- {gs.cv_results_['std_test_score'][i]:.4f}",
                      "Num. Candidates": len(gs.cv_results_["params"]), "Search Time (s)": round(time.time() - t, 1)})
    print(name, "->", best, f"({time.time()-t:.0f}s)")
tune_df = pd.DataFrame(tune_rows).set_index("Model"); display(tune_df)
save_table(tune_df, "Ext01_ensemble_hyperparameters", caption="Hyperparameters of the ensemble models (subject-grouped 5-fold CV on training subjects).")
ALL_MODELS = list(MODELS)

In [ ]:
preds, rows = {}, []
for name in ALL_MODELS:
    t = time.time(); m = clone(MODELS[name]).fit(X_train, y_train); fit_s = time.time() - t
    t = time.time(); yp = m.predict(X_test); inf_ms = (time.time() - t) / len(y_test) * 1000
    preds[name] = yp
    p, r, f, _ = precision_recall_fscore_support(y_test, yp, average="macro", zero_division=0)
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, yp), "Balanced Acc.": balanced_accuracy_score(y_test, yp),
                 "Precision (macro)": p, "Recall (macro)": r, "F1 (macro)": f, "MCC": matthews_corrcoef(y_test, yp),
                 "Train Acc.": accuracy_score(y_train, m.predict(X_train)), "Training Time (s)": fit_s, "Inference (ms/sample)": inf_ms})
test_df_res = pd.DataFrame(rows).set_index("Model")
test_df_res["Gap (Train-Test)"] = test_df_res["Train Acc."] - test_df_res["Accuracy"]
display(test_df_res.round(4))
save_table(test_df_res, "Ext02_test_results_with_ensembles", caption="Test-set results including the ensemble baselines.")

print("\nReproducibility check against the main run (test accuracy):")
for k, v in EXPECTED_TEST_ACC.items():
    d = test_df_res.loc[k, "Accuracy"] - v
    print(f"  {k:14s} now={test_df_res.loc[k,'Accuracy']:.4f} main={v:.4f} diff={d:+.4f}", "OK" if abs(d) < 0.003 else "<-- WARNING: differs from main run")

pd.DataFrame({"subject": g_test, "y_true": le.inverse_transform(y_test), **{n: le.inverse_transform(p) for n, p in preds.items()}}
             ).to_csv(OUT_DIR / "tables" / "Ext_test_predictions.csv", index=False)

In [ ]:
# Pairwise McNemar tests: Linear SVM versus every other model (Holm-corrected)
base = "SVM (Linear)"; rows = []
for other in [m for m in ALL_MODELS if m != base]:
    n01, n10, p = mcnemar_p(preds[base] == y_test, preds[other] == y_test)
    rows.append({"Model A": base, "Model B": other, "A correct / B wrong": n01, "A wrong / B correct": n10, "p": p})
mc = pd.DataFrame(rows); mc["p (Holm)"] = holm(mc["p"].values)
mc["Significance"] = np.where(mc["p (Holm)"] < 0.001, "***", np.where(mc["p (Holm)"] < 0.01, "**", np.where(mc["p (Holm)"] < 0.05, "*", "ns")))
display(mc.round(4)); save_table(mc.set_index(["Model A", "Model B"]), "Ext03_mcnemar_vs_linear_svm")

## B. Subject-Level (Cluster) Bootstrap Confidence Intervals

Windows of the same subject are strongly correlated, so a window-level bootstrap gives intervals that are too narrow.
Here the **9 test subjects are resampled with replacement** (all windows of a drawn subject are kept together).

In [ ]:
subs = np.unique(g_test); idx_by = {s: np.where(g_test == s)[0] for s in subs}
rng = np.random.default_rng(SEED)
boot = {m: {"acc": [], "f1": []} for m in ALL_MODELS}; diffs = {m: [] for m in ALL_MODELS if m != base}
for b in range(N_BOOT):
    draw = rng.choice(subs, size=len(subs), replace=True)
    idx = np.concatenate([idx_by[s] for s in draw]); yt = y_test[idx]
    accs = {}
    for m in ALL_MODELS:
        yp = preds[m][idx]; a = float((yp == yt).mean()); accs[m] = a
        boot[m]["acc"].append(a); boot[m]["f1"].append(f1_score(yt, yp, average="macro", zero_division=0))
    for m in diffs: diffs[m].append(accs[base] - accs[m])

rows = []
for m in ALL_MODELS:
    a, f = np.array(boot[m]["acc"]), np.array(boot[m]["f1"])
    rows.append({"Model": m, "Accuracy": test_df_res.loc[m, "Accuracy"], "Acc 95% CI low": np.percentile(a, 2.5), "Acc 95% CI high": np.percentile(a, 97.5),
                 "F1 (macro)": test_df_res.loc[m, "F1 (macro)"], "F1 95% CI low": np.percentile(f, 2.5), "F1 95% CI high": np.percentile(f, 97.5)})
ci_df = pd.DataFrame(rows).set_index("Model"); display(ci_df.round(4))
save_table(ci_df, "Ext04_subject_level_bootstrap_ci", caption="Test-set metrics with subject-level (cluster) bootstrap 95% CIs.")

rows = []
for m, d in diffs.items():
    d = np.array(d)
    rows.append({"Comparison": f"{base} - {m} (accuracy)", "Mean diff": d.mean(), "95% CI low": np.percentile(d, 2.5),
                 "95% CI high": np.percentile(d, 97.5), "Share of resamples with diff > 0": float((d > 0).mean())})
diff_df = pd.DataFrame(rows).set_index("Comparison"); display(diff_df.round(4))
save_table(diff_df, "Ext05_subject_level_paired_differences", caption="Subject-level bootstrap of paired accuracy differences (Linear SVM minus other models).")

fig, ax = plt.subplots(figsize=(7, 3.8)); order = ci_df.sort_values("Accuracy").index
for i, m in enumerate(order):
    r = ci_df.loc[m]; ax.errorbar(r["Accuracy"] * 100, i, xerr=[[ (r["Accuracy"] - r["Acc 95% CI low"]) * 100], [(r["Acc 95% CI high"] - r["Accuracy"]) * 100]],
                                  fmt="o", capsize=4, color="tab:blue")
ax.set_yticks(range(len(order))); ax.set_yticklabels(order); ax.set_xlabel("Test accuracy (%) with subject-level 95% CI")
ax.set_title("Test accuracy with subject-level bootstrap CIs"); savefig("FigureExt01_subject_level_bootstrap_ci")

## C. Repeated Subject-Grouped K-Fold CV on All 30 Subjects

Hyperparameters are fixed (Section 2 / A; they were selected on the 21 training subjects only), so this is a stability
analysis rather than a nested CV. Each repetition uses a different random subject-to-fold assignment.

In [ ]:
def run_fold(name, tr, te, rep, fold):
    m = clone(MODELS[name]).fit(X_all[tr], y_all[tr]); yp = m.predict(X_all[te])
    return {"Model": name, "Repeat": rep, "Fold": fold, "Accuracy": accuracy_score(y_all[te], yp),
            "F1 (macro)": f1_score(y_all[te], yp, average="macro", zero_division=0)}

if RUN_REPEATED_CV:
    t = time.time(); jobs = []
    for rep in range(N_REPEATS):
        cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + rep)
        for fold, (tr, te) in enumerate(cv.split(X_all, y_all, g_all)):
            for name in ALL_MODELS: jobs.append((name, tr, te, rep, fold))
    res = Parallel(n_jobs=-1)(delayed(run_fold)(*j) for j in jobs)
    rep_df = pd.DataFrame(res); print(f"{len(jobs)} fits in {time.time()-t:.0f}s")
    save_table(rep_df.set_index(["Model", "Repeat", "Fold"]), "Ext06_repeated_cv_fold_scores")
    summ = rep_df.groupby("Model").agg(**{"Accuracy mean": ("Accuracy", "mean"), "Accuracy std": ("Accuracy", "std"),
                                           "F1 mean": ("F1 (macro)", "mean"), "F1 std": ("F1 (macro)", "std")}).loc[ALL_MODELS]
    per_rep = rep_df.groupby(["Model", "Repeat"])["Accuracy"].mean().unstack()
    summ["Std of repeat means"] = per_rep.std(axis=1).loc[ALL_MODELS]
    display(summ.round(4)); save_table(summ, "Ext07_repeated_cv_summary", caption=f"{N_REPEATS}x{N_FOLDS}-fold subject-grouped CV (all 30 subjects).")

    # Friedman + Wilcoxon (Holm) on repeat 0 folds (independent folds, as in the main study)
    piv = rep_df[rep_df.Repeat == 0].pivot(index="Fold", columns="Model", values="Accuracy")[ALL_MODELS]
    chi2, pf = stats.friedmanchisquare(*[piv[m] for m in ALL_MODELS])
    print(f"Friedman (repeat 0, {len(piv)} folds, {len(ALL_MODELS)} models): chi2={chi2:.3f}, p={pf:.2e}")
    rows = []
    for other in [m for m in ALL_MODELS if m != base]:
        try: w, p = stats.wilcoxon(piv[base], piv[other])
        except ValueError: w, p = np.nan, 1.0
        rows.append({"Model A": base, "Model B": other, "Mean diff": (piv[base] - piv[other]).mean(), "W": w, "p": p})
    wdf = pd.DataFrame(rows); wdf["p (Holm)"] = holm(wdf["p"].values); display(wdf.round(4))
    save_table(wdf.set_index(["Model A", "Model B"]), "Ext08_wilcoxon_vs_linear_svm_repeat0")
    pd.DataFrame({"Friedman chi2": [chi2], "p": [pf], "Folds": [len(piv)]}).to_csv(OUT_DIR / "tables" / "Ext08b_friedman_repeat0.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=rep_df, x="Model", y="Accuracy", order=ALL_MODELS, ax=ax, palette="Set2", width=0.55)
    sns.stripplot(data=rep_df, x="Model", y="Accuracy", order=ALL_MODELS, ax=ax, color="k", size=2.5, alpha=0.5)
    ax.set_title(f"{N_REPEATS}x{N_FOLDS}-fold subject-independent CV (all 30 subjects)"); plt.xticks(rotation=15)
    savefig("FigureExt02_repeated_cv_boxplot")
else:
    print("Repeated CV skipped (RUN_REPEATED_CV = False).")

## D. Leave-One-Subject-Out (LOSO) Evaluation

Each of the 30 subjects is held out once. Gradient Boosting is excluded from LOSO for run-time reasons.

In [ ]:
LOSO_MODELS = [m for m in ALL_MODELS if m != "Gradient Boosting"]

def loso_fold(name, tr, te, subj):
    m = clone(MODELS[name]).fit(X_all[tr], y_all[tr]); yp = m.predict(X_all[te])
    return {"Model": name, "Subject": int(subj), "Accuracy": accuracy_score(y_all[te], yp), "n": len(te)}

if RUN_LOSO:
    t = time.time(); jobs = []
    for tr, te in LeaveOneGroupOut().split(X_all, y_all, g_all):
        for name in LOSO_MODELS: jobs.append((name, tr, te, g_all[te][0]))
    loso_df = pd.DataFrame(Parallel(n_jobs=-1)(delayed(loso_fold)(*j) for j in jobs)); print(f"{len(jobs)} fits in {time.time()-t:.0f}s")
    wide = loso_df.pivot(index="Subject", columns="Model", values="Accuracy")[LOSO_MODELS]
    save_table(wide, "Ext09_loso_per_subject_accuracy", caption="LOSO accuracy for each held-out subject.")
    ls = pd.DataFrame({"Mean": wide.mean(), "Std": wide.std(), "Min": wide.min(), "Worst subject": wide.idxmin(), "Median": wide.median(), "Max": wide.max()})
    display(ls.round(4)); save_table(ls, "Ext10_loso_summary", caption="LOSO accuracy summary (30 held-out subjects).")
    chi2, pf = stats.friedmanchisquare(*[wide[m] for m in LOSO_MODELS])
    print(f"Friedman over {len(wide)} subjects: chi2={chi2:.3f}, p={pf:.2e}")
    rows = []
    for other in [m for m in LOSO_MODELS if m != base]:
        try: w, p = stats.wilcoxon(wide[base], wide[other])
        except ValueError: w, p = np.nan, 1.0
        rows.append({"Model A": base, "Model B": other, "Mean diff": (wide[base] - wide[other]).mean(), "W": w, "p": p})
    wl = pd.DataFrame(rows); wl["p (Holm)"] = holm(wl["p"].values); display(wl.round(4))
    save_table(wl.set_index(["Model A", "Model B"]), "Ext11_loso_wilcoxon_vs_linear_svm")
    pd.DataFrame({"Friedman chi2": [chi2], "p": [pf], "Subjects": [len(wide)]}).to_csv(OUT_DIR / "tables" / "Ext11b_loso_friedman.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=loso_df, x="Model", y="Accuracy", order=LOSO_MODELS, ax=ax, palette="Set2", width=0.55)
    sns.stripplot(data=loso_df, x="Model", y="Accuracy", order=LOSO_MODELS, ax=ax, color="k", size=3, alpha=0.6)
    ax.set_title("Leave-one-subject-out accuracy (each point = one subject)"); plt.xticks(rotation=15)
    savefig("FigureExt03_loso_per_subject")
else:
    print("LOSO skipped (RUN_LOSO = False).")

## E. Confusion Matrices and Per-Class Results (raw tables for the manuscript figure)

In [ ]:
rows = []
for m in ALL_MODELS:
    cm = confusion_matrix(y_test, preds[m], labels=range(K))
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(OUT_DIR / "tables" / f"Ext12_confusion_counts_{m.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
    p, r, f, s = precision_recall_fscore_support(y_test, preds[m], labels=range(K), zero_division=0)
    for i, c in enumerate(class_names): rows.append({"Model": m, "Class": c, "Precision": p[i], "Recall": r[i], "F1": f[i], "Support": int(s[i])})
pc = pd.DataFrame(rows).set_index(["Model", "Class"]); save_table(pc, "Ext13_per_class_results")
display(pc["F1"].unstack().round(3))

n = len(ALL_MODELS); cols = 3; rws = int(np.ceil(n / cols))
fig, axes = plt.subplots(rws, cols, figsize=(5.2 * cols, 4.6 * rws)); axes = np.array(axes).ravel()
for ax, m in zip(axes, ALL_MODELS):
    cm = confusion_matrix(y_test, preds[m], labels=range(K), normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=ax, xticklabels=class_names, yticklabels=class_names, vmin=0, vmax=1)
    ax.set_title(f"{m} (Acc={accuracy_score(y_test, preds[m]):.3f})"); ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.tick_params(axis="x", rotation=35)
for ax in axes[n:]: ax.axis("off")
plt.tight_layout(); savefig("FigureExt04_confusion_matrices_normalized")

## F. Save and Download All Outputs

In [ ]:
cfg = {"env": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
               "scikit-learn": sklearn.__version__, "scipy": scipy.__version__, "seed": SEED},
       "settings": {"FAST_MODE": FAST_MODE, "N_FOLDS": N_FOLDS, "N_REPEATS": N_REPEATS, "N_BOOT": N_BOOT,
                    "RF_TREES": RF_TREES, "HGB_ITER": HGB_ITER, "RUN_REPEATED_CV": RUN_REPEATED_CV, "RUN_LOSO": RUN_LOSO},
       "model_params": {n: {k: str(v) for k, v in m.named_steps["clf"].get_params().items() if k in
                            ("C", "gamma", "kernel", "n_neighbors", "weights", "metric", "criterion", "max_depth", "min_samples_leaf",
                             "n_estimators", "max_features", "learning_rate", "l2_regularization", "max_iter")}
                        for n, m in MODELS.items()},
       "runtime_seconds": round(time.time() - T0, 1)}
(OUT_DIR / "config_and_environment_extension.json").write_text(json.dumps(cfg, indent=2))
zip_path = shutil.make_archive("har_extension_outputs", "zip", OUT_DIR)
print("ZIP:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)  total runtime {time.time()-T0:.0f}s")
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Not running in Colab; download the ZIP manually.")